# MANTIS v8 — Simulation de contrôle EMG continu (avec repos)

**Objectif** : passer du classifieur de fenêtres à un système de commande qui peut aussi **ne rien faire**. On teste une chaîne continue : **signal → fenêtre de 0,5 s toutes les 100 ms → LDA général + calibration personnelle kNN → moyenne causale → décision stable → mesures de faux déclenchements et de délai**.

### En quoi cette v8 diffère de la v7 ?

- Les fenêtres sont reconstruites sur le **signal continu**, y compris les portions `restimulus = 0` (repos) et les **transitions** ; le code de décision n'a jamais accès aux vrais labels ni aux débuts/fins annotés.
- Les décisions se poursuivent à travers plusieurs gestes. L'état est remis à zéro **uniquement aux frontières d'enregistrement** (changement de sujet, exercice, discontinuité du CSV), jamais à chaque nouveau geste.
- Les nouvelles mesures incluent **rappel des gestes par événement**, **faux déclenchements par heure de repos**, **temps d'observation depuis l'annotation du début du geste**, accuracy et macro-F1 ; toutes ont des limitations expliquées.
- Les hyperparamètres sont choisis sur **19 participants historiques de recherche** par validation croisée par groupes. Les 4 participants de validation historiques permettent une vérification *exploratoire*. Les 4 anciens participants test (`12,13,16,25`) restent exclus du choix et de l'évaluation.

**Limites à lire avant de comparer les chiffres :** il s'agit d'un **rejeu hors ligne**, non d'une application branchée à un capteur. En particulier, l'adaptation `répétitions 1–2` / évaluation `3–10` forme un protocole de calibration *rétrospective* : le CSV n'a pas forcément enregistré toute la calibration de chaque geste avant toutes ses nouvelles répétitions. Les annotations servent à définir les données de calibration et à mesurer les performances, **jamais à guider la prédiction**. Les résultats ne sont pas une latence physique ni une preuve prospective sur des utilisateurs complètement inédits.

**Ne pas confondre** le contrôle du repos avec les 52 gestes : la v8 doit reconnaître **53 états** (repos `0` + gestes `1…52`).

## 1 — Configuration (Kaggle)

LDA et kNN fonctionnent sur **CPU**. Le GPU n'est pas nécessaire pour cette expérience. Un nouveau cache indépendant de celui de la v7 est créé : **la v7 supprimait le repos**, donc son cache ne serait pas valide ici.

In [ ]:
from pathlib import Path
import time, json, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, accuracy_score

SEED = 42
SPLIT_SEED = 137
SR = 100
WINDOW = 50            # 0.5 seconde de signal EMG
HOP = 10               # 0.1 seconde entre deux prédictions
CHUNK_SIZE = 150_000
N_CHANNELS = 10
N_CLASSES = 53          # 0=REPOS et 1..52=gestes
CLASSES = np.arange(N_CLASSES, dtype=np.int16)
EMG_COLS = [f'emg_{i}' for i in range(N_CHANNELS)]
OFFSET = {1:0, 2:12, 3:29}
MAX_TRAIN_PER_SUBJECT_CLASS = 70
MAX_CAL_PER_CLASS = 10
RESEARCH_FOLDS = 3
CAL_BUDGETS = [0, 2]
ALPHAS_PERSONAL = [0.50, 0.75, 1.00]
SMOOTH_K = [1, 3, 5]
CONF_THRESHOLDS = [0.35, 0.60, 0.85]
ACTIVE_STABILITY = 2    # 2 prédictions similaires avant d'activer un geste
REST_STABILITY = 2
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
np.random.seed(SEED)
print('V8 | 53 états | fenêtre:', WINDOW/SR, 's | pas:', HOP/SR, 's | device: CPU (LDA/kNN)')

## 2 — Attacher le CSV original

Le notebook **ne contient pas le CSV de 3,37 Go**. Sur Kaggle, il cherche d'abord le dataset attaché puis utilise `kagglehub` si nécessaire.

In [ ]:
DATASET_HANDLE='mansibmursalin/ninapro-db1-full-dataset'
DATASET_FILENAME='Ninapro_DB1.csv'
candidates=[
    Path('/kaggle/input/datasets/mansibmursalin/ninapro-db1-full-dataset/Ninapro_DB1.csv'),
    Path('/kaggle/input/ninapro-db1-full-dataset/Ninapro_DB1.csv'),
]
DATASET_PATH = next((p for p in candidates if p.is_file()), None)
if DATASET_PATH is None:
    import kagglehub
    downloaded = Path(kagglehub.dataset_download(DATASET_HANDLE, path=DATASET_FILENAME))
    if downloaded.is_file():
        DATASET_PATH = downloaded
    else:
        candidates_found = list(downloaded.rglob(DATASET_FILENAME))
        if len(candidates_found) != 1: raise FileNotFoundError('CSV NinaPro DB1 non trouvé.')
        DATASET_PATH = candidates_found[0]
assert DATASET_PATH.is_file()
print('Fichier:',DATASET_PATH,'| Go:',round(DATASET_PATH.stat().st_size/1e9,2))

## 3 — Extraire des caractéristiques **sans couper aux transitions**

**C'est le changement méthodologique le plus important.** Chaque enregistrement est un bloc continu par `(participant, exercice)` ; les labels peuvent changer au sein de ce bloc, et cela ne réinitialise pas le modèle. La fonction ci-dessous ne découpe jamais une fenêtre entre deux participants ou deux enregistrements non contigus.

Pour limiter la mémoire, nous gardons **60 caractéristiques** par fenêtre plutôt que les 50×10 mesures brutes pour chaque prédiction à 100 ms. Les caractéristiques sont RMS, MAV, écart-type, longueur d'onde moyenne, maximum absolu et RMS relatif, sur les 10 canaux. Ce choix d'ingénierie change la représentation par rapport à la v7 et doit être pris en compte dans les comparaisons.

Les annotations (`restimulus`, `rerepetition`) servent à construire les **cibles**, les groupes de calibration et les **statistiques a posteriori**. Elles ne sont **jamais** transmises à la fonction de décision.

In [ ]:
FEATURE_NAMES = [f'{kind}_emg_{c}' for kind in (
    'rms','mav','std','waveform_length','max_abs','relative_rms') for c in range(10)]

def features_windows(signal, starts, block=6000):
    from numpy.lib.stride_tricks import sliding_window_view
    # Vue (n_fenêtres_possibles, canaux, temps), sans copie du signal.
    all_views = sliding_window_view(signal, window_shape=WINDOW, axis=0)
    result = np.empty((len(starts),60),dtype=np.float32)
    for offset in range(0,len(starts),block):
        sl=slice(offset,offset+block)
        x=all_views[starts[sl]]  # (batch,10,50)
        absx=np.abs(x)
        rms=np.sqrt(np.mean(x*x,axis=2)+1e-12)
        mav=np.mean(absx,axis=2)
        std=np.std(x,axis=2)
        wl=np.mean(np.abs(np.diff(x,axis=2)),axis=2)
        maximum=np.max(absx,axis=2)
        relative=rms/(rms.sum(axis=1,keepdims=True)+1e-8)
        result[sl]=np.concatenate([rms,mav,std,wl,maximum,relative],axis=1)
    return result


def forward_fill_repeat(rep):
    """Attribue au repos la répétition du geste voisin, uniquement pour les partitions."""
    good=np.flatnonzero(rep>0)
    if not len(good): return np.zeros(len(rep),dtype=np.int16)
    ix=np.maximum.accumulate(np.where(rep>0,np.arange(len(rep)),-1))
    return np.where(ix>=0,rep[np.maximum(ix,0)],rep[good[0]]).astype(np.int16)


def extract_continuous_stream(csv_path, chunk_size=150_000):
    """Lecture en streaming du CSV; étiquette 0 conservée; état non coupé par geste."""
    cols=pd.read_csv(csv_path,nrows=0).columns.tolist()
    has_idx='Unnamed: 0' in cols
    chosen=EMG_COLS+['subject','exercise','restimulus','rerepetition']+(['Unnamed: 0'] if has_idx else [])
    missing=set(chosen)-set(cols)
    if missing: raise ValueError('Colonnes manquantes: '+str(sorted(missing)))
    dtypes={c:'float32' for c in EMG_COLS}
    dtypes.update({c:'int16' for c in ['subject','exercise','restimulus','rerepetition']})
    if has_idx:dtypes['Unnamed: 0']='int64'
    block_features=[]; block_labels=[];block_subject=[];block_phase=[];block_pure=[]
    block_session=[];block_event=[];block_since=[];block_sample_idx=[];block_phase_start=[]
    event_rows=[]; pending_key=None; pending_end=None; pending_parts=[]
    session_id=0;event_counter=0; row_offset=0; bad_rows=0

    def flush():
        nonlocal pending_key,pending_end,pending_parts,session_id,event_counter
        if pending_key is None: return
        person,exercise=pending_key
        x=np.concatenate([q[0] for q in pending_parts],axis=0)
        stim=np.concatenate([q[1] for q in pending_parts])
        rep=np.concatenate([q[2] for q in pending_parts])
        start_global=pending_parts[0][3]
        if not np.all((stim>=0)&(stim<=({1:12,2:17,3:23}[exercise]))):
            raise ValueError('Label incorrect, exercice '+str(exercise))
        label=np.where(stim>0,stim+OFFSET[exercise],0).astype(np.int16)
        phase=forward_fill_repeat(rep)
        changes=(label[1:]!=label[:-1])|((label[1:]>0)&(rep[1:]!=rep[:-1]))
        ev_starts=np.r_[0,np.flatnonzero(changes)+1]
        ev_ends=np.r_[ev_starts[1:],len(x)]
        local_ev=np.searchsorted(ev_starts,np.arange(len(x)),side='right')-1
        for k,(a,b) in enumerate(zip(ev_starts,ev_ends)):
            event_rows.append({'event_id':event_counter+k,'session':session_id,
                'subject':person,'exercise':exercise,'gesture':int(label[a]),
                'phase':int(phase[a]),'duration_s':(b-a)/SR,'start_row':int(start_global+a),
                'end_row':int(start_global+b-1)})
        if len(x)>=WINDOW:
            starts=np.arange(0,len(x)-WINDOW+1,HOP,dtype=np.int64)
            ends=starts+WINDOW-1
            feat=features_windows(x,starts)
            block_features.append(feat)
            block_labels.append(label[ends])
            block_subject.append(np.full(len(starts),person,dtype=np.int16))
            block_phase.append(phase[ends])
            block_phase_start.append(phase[starts])
            block_pure.append((local_ev[starts]==local_ev[ends]) & (phase[starts]==phase[ends]))
            block_session.append(np.full(len(starts),session_id,dtype=np.int32))
            block_event.append((local_ev[ends]+event_counter).astype(np.int32))
            block_since.append(((ends-ev_starts[local_ev[ends]])/SR).astype(np.float32))
            block_sample_idx.append((ends+start_global).astype(np.int64))
        event_counter+=len(ev_starts)
        session_id+=1;pending_key=None;pending_end=None;pending_parts=[]

    t0=time.time()
    for ci,df in enumerate(pd.read_csv(csv_path,usecols=chosen,dtype=dtypes,chunksize=chunk_size),1):
        n=len(df)
        if n==0:continue
        sig=df[EMG_COLS].to_numpy(dtype=np.float32,copy=True)
        subj=df.subject.to_numpy(dtype=np.int16)
        ex=df.exercise.to_numpy(dtype=np.int16)
        st=df.restimulus.to_numpy(dtype=np.int16)
        rp=df.rerepetition.to_numpy(dtype=np.int16)
        ix=(df['Unnamed: 0'].to_numpy(dtype=np.int64) if has_idx else
            np.arange(row_offset,row_offset+n,dtype=np.int64))
        row_offset+=n
        finite=np.isfinite(sig).all(axis=1)
        bad_rows+=int((~finite).sum())
        breaks=(subj[1:]!=subj[:-1])|(ex[1:]!=ex[:-1])|(ix[1:]!=ix[:-1]+1)|(~finite[1:])|(~finite[:-1])
        bounds=np.r_[0,np.flatnonzero(breaks)+1,n]
        for a,b in zip(bounds[:-1],bounds[1:]):
            if not finite[a]:
                flush();continue
            key=(int(subj[a]),int(ex[a]))
            if key[1] not in OFFSET:raise ValueError('Exercice inconnu: '+str(key))
            if pending_key!=key or (pending_end is not None and int(ix[a])!=pending_end+1):
                flush()
                pending_key=key
            pending_parts.append((sig[a:b],st[a:b],rp[a:b],int(ix[a])))
            pending_end=int(ix[b-1])
            if b<n:flush()
        if ci%20==0:print(f'{row_offset:,} lignes | {session_id} sessions | {time.time()-t0:.0f}s')
    flush()
    if not block_features:raise RuntimeError('Aucune fenêtre possible')
    keys=['features','label','subject','phase','pure','session','event_id','since_onset_s','endpoint_row','phase_start']
    arrays=[block_features,block_labels,block_subject,block_phase,block_pure,
            block_session,block_event,block_since,block_sample_idx,block_phase_start]
    out={k:np.concatenate(v) for k,v in zip(keys,arrays)}
    events=pd.DataFrame(event_rows)
    print(f'Extraction terminée: {row_offset:,} lignes, {len(out["label"]):,} fenêtres, {len(events):,} événements, {bad_rows} lignes invalides')
    return out,events

## 4 — Construire et réutiliser un cache **spécifique v8**

Sur le CSV DB1, cela demande une lecture complète. Ne jamais utiliser le cache v7 : il ne contient ni le repos ni les fenêtres de transition.

In [ ]:
CACHE_PATH=WORK_DIR/f'mantis_v8_stream_50_10_rest_bytes{DATASET_PATH.stat().st_size}.npz'
EVENT_PATH=WORK_DIR/f'mantis_v8_events_bytes{DATASET_PATH.stat().st_size}.csv'
required_cache_keys={'features','label','subject','phase','phase_start','pure',
                     'session','event_id','since_onset_s','endpoint_row'}
cache_valid=False
if CACHE_PATH.is_file() and EVENT_PATH.is_file():
    with np.load(CACHE_PATH,allow_pickle=False) as npz:
        cache_valid=required_cache_keys.issubset(set(npz.files))
        if cache_valid:
            d={k:npz[k] for k in npz.files}
    if cache_valid:
        print('Chargement cache v8:',CACHE_PATH)
        events=pd.read_csv(EVENT_PATH)
    else:
        print('Cache obsolète : reconstruction nécessaire')
if not cache_valid:
    d,events=extract_continuous_stream(DATASET_PATH,CHUNK_SIZE)
    np.savez(CACHE_PATH,**d) # non compressé pour accélérer les futurs redémarrages
    events.to_csv(EVENT_PATH,index=False)
features=d['features']; labels=d['label'];subjects=d['subject'];phase=d['phase']; phase_start=d['phase_start']
pure=d['pure'];sessions=d['session'];event_ids=d['event_id'];since_s=d['since_onset_s']
assert features.shape==(len(labels),60)
assert np.isfinite(features).all()
assert np.all(np.isin(labels,CLASSES))
print('Fenêtres:',len(labels),'repos:',round(np.mean(labels==0)*100,1),'% | classes:',len(np.unique(labels)))
print('Participants:',len(np.unique(subjects)),'| sessions:',len(np.unique(sessions)))
display(events.groupby('gesture').agg(n_events=('event_id','size'),duration_s=('duration_s','sum')).head())

## 5 — Répartition des personnes et de la calibration

**Général :** le modèle LDA est entraîné sur des personnes différentes des personnes sur lesquelles il est évalué. **Personnel :** pour une nouvelle personne, les fenêtres **pures** des répétitions 1 et 2 (et du repos attribué à cette période) peuvent entraîner le kNN. **Évaluation :** répétitions 3 à 10 et repos associé. Les fenêtres qui chevauchent une limite de période ne sont jamais utilisées comme exemples d'entraînement.

**Point de méthode** : les répétitions de NinaPro ne constituent pas nécessairement un bloc de calibration acquis intégralement avant le replay. L'expérience personnelle est donc rétrospective. Pour la validation vraiment prospective, il faudra enregistrer notre propre séance continue après calibration.

Les anciennes personnes test sont exclues de tout réglage : elles ont déjà été consultées dans le passé.

In [ ]:
sids=np.unique(subjects)
shuffled=np.random.default_rng(SPLIT_SEED).permutation(sids)
HIST_TEST=np.sort(shuffled[:4]); HIST_VAL=np.sort(shuffled[4:8]); RESEARCH=np.sort(shuffled[8:])
assert np.array_equal(HIST_TEST,[12,13,16,25])
assert np.array_equal(HIST_VAL,[4,19,24,27])
assert len(RESEARCH)==19
print('Recherche/CV 19 sujets:',RESEARCH.tolist())
print('Validation historique exploratoire:',HIST_VAL.tolist())
print('Anciens tests EXCLUS:',HIST_TEST.tolist())
print('Repos inclus:',int((labels==0).sum()),'fenêtres')


def capped_train_indices(person_set, max_per=MAX_TRAIN_PER_SUBJECT_CLASS):
    selected=[]
    for person in person_set:
        person_inds=np.flatnonzero((subjects==person)&pure)
        for lab in CLASSES:
            inds=person_inds[labels[person_inds]==lab]
            if len(inds)>max_per:
                inds=np.random.default_rng(SEED+int(person)*103+int(lab)).choice(inds,max_per,replace=False)
            selected.append(inds)
    return np.sort(np.concatenate(selected)) if selected else np.empty(0,dtype=np.int64)


def calibration_indices(person,budget):
    mask=(subjects==person)&pure&(phase>=1)&(phase<=budget)
    selection=[]
    for lab in CLASSES:
        ix=np.flatnonzero(mask&(labels==lab))
        if len(ix)>MAX_CAL_PER_CLASS:
            ix=np.random.default_rng(SEED+int(person)*1009+int(lab)).choice(ix,MAX_CAL_PER_CLASS,replace=False)
        selection.append(ix)
    return np.sort(np.concatenate(selection)) if selection else np.empty(0,dtype=np.int64)


def evaluation_mask(person):
    return (subjects==person)&(phase>=3)&(phase<=10)&(phase_start>=3)&(phase_start<=10)

# Contrôles sur les fenêtres retenues.
for person in RESEARCH[:3]:
    c=calibration_indices(int(person),2)
    assert np.all((phase[c]<=2)&(labels[c]>=0)&pure[c])
    assert not np.any(evaluation_mask(int(person))[c])
print('Séparation calibration / évaluation : vérifiée')

## 6 — Modèle général LDA et calibration kNN

Les deux modèles sont **distincts**. LDA est entraîné à partir d'autres participants. Le kNN personnel ne voit que les répétitions autorisées de la nouvelle personne. La classe repos (`0`) est incluse dans les deux. Tous les modèles alignent leurs scores sur les 53 classes.

Un poids `alpha=0.75` signifie 75 % kNN personnel et 25 % LDA général. Les paramètres seront sélectionnés sur les participants de recherche, pas sur les anciens tests.

In [ ]:
def create_general_model(train_people):
    ix=capped_train_indices(train_people)
    model=make_pipeline(StandardScaler(),
                        LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto'))
    model.fit(features[ix],labels[ix])
    return model


def probabilities_53(raw,classes):
    out=np.zeros((len(raw),53),dtype=np.float32)
    out[:,np.asarray(classes,dtype=int)]=raw
    return out


def get_scores(general, person):
    # Le modèle reçoit TOUT le flux de la personne, y compris repos et calibrations.
    ix=np.flatnonzero(subjects==person)
    if not len(ix):raise ValueError('Participant absent')
    assert np.all(np.diff(sessions[ix])>=0)
    pg=probabilities_53(general.predict_proba(features[ix]),
                        general.classes_)
    cidx=calibration_indices(person,2)
    if len(cidx):
        scaler=general.named_steps['standardscaler']
        knn=KNeighborsClassifier(n_neighbors=min(3,len(cidx)),weights='distance',n_jobs=-1)
        knn.fit(scaler.transform(features[cidx]),labels[cidx])
        pp=probabilities_53(knn.predict_proba(scaler.transform(features[ix])),knn.classes_)
    else:
        pp=None
    return ix,pg,pp,{'n_cal':len(cidx),'classes_cal':len(np.unique(labels[cidx]))}

## 7 — Décideur temporel **sans accès à la vérité**

La fonction `continuous_decisions` **ne reçoit que** des scores de classes et des identifiants de sessions d'enregistrement. Elle ne voit ni `labels`, ni `event_id`, ni `phase`, ni `repetitions`.

À chaque pas de 100 ms : moyenne causale des `K` dernières prédictions (moins de `K` au démarrage), seuil minimal de confiance, puis 2 décisions concordantes avant d'activer une commande. Un repos stable désactive la commande. Aucun reset au début ou à la fin d'un geste. **Le modèle doit apprendre quand se taire.**

In [ ]:
def continuous_decisions(probs,session_ids,k=3,threshold=.35,
                         active_stability=ACTIVE_STABILITY, rest_stability=REST_STABILITY):
    """Aucune entrée de type label, trial_id, repetition ou phase n'est autorisée."""
    assert len(probs)==len(session_ids) and probs.shape[1]==53
    out=np.zeros(len(probs),dtype=np.int16)
    boundaries=np.r_[0,np.flatnonzero(session_ids[1:]!=session_ids[:-1])+1,len(probs)]
    for a,b in zip(boundaries[:-1],boundaries[1:]):
        p=probs[a:b]
        cs=np.vstack([np.zeros((1,p.shape[1]),dtype=np.float64),np.cumsum(p,axis=0,dtype=np.float64)])
        j=np.arange(len(p))
        left=np.maximum(0,j+1-k)
        mean=(cs[j+1]-cs[left])/(j+1-left)[:,None]
        winner=np.argmax(mean,axis=1)
        confidence=mean[j,winner]
        proposed=np.where(confidence>=threshold,winner,0).astype(np.int16)
        state=0;candidate=0;streak=0
        for t,new in enumerate(proposed):
            new=int(new)
            if new==candidate:streak+=1
            else:candidate=new;streak=1
            needed=rest_stability if candidate==0 else active_stability
            if streak>=needed: state=candidate
            out[a+t]=state
    return out

# Tests de causalité et d'absence de reset par geste.
test_probs=np.zeros((8,53),dtype=np.float32)
test_probs[:4,1]=.9;test_probs[:4,0]=.1
test_probs[4:,0]=.9;test_probs[4:,1]=.1
before=continuous_decisions(test_probs,np.zeros(8,dtype=int),k=2,threshold=.35)
modified=test_probs.copy();modified[6:,2]=1;modified[6:,0]=0
assert np.array_equal(continuous_decisions(modified,np.zeros(8,dtype=int),k=2,threshold=.35)[:6],before[:6]),'Fuite du futur'
assert before[1]==1 and before[-1]==0
print('Tests: causalité, repos, hystérésis et transitions : OK')

## 8 — Évaluer les événements et les fausses commandes

Nous évaluons **après** la prédiction en utilisant les annotations du CSV. Il est normal que cette fonction utilise les vrais labels : **elle mesure** la qualité et ne produit aucune décision. Le rappel par événement inclut les gestes trop courts pour créer une fenêtre et les gestes jamais reconnus.

- **Rappel événementiel** : part des gestes annotés pour lesquels la commande correcte a été émise au moins une fois pendant le geste.
- **Faux déclenchements/heure de repos** : nouvelles périodes avec une commande non nulle alors que le repos est annoté (normalisé par la durée totale de repos).
- **Temps d'observation** : délai entre le début annoté du geste et le premier endpoint où la bonne commande est émise. Ne mesure pas le temps de calcul, ni la latence matérielle.
- **Macro-F1 actif** : sur les fenêtres éligibles dont l'annotation est un geste, avec les 52 classes.

Un label de fin de fenêtre peut différer de son contenu pendant une transition : ce délai est une difficulté réelle du contrôle continu.

In [ ]:
events_indexed=events.set_index('event_id',drop=False)

def evaluate_commands(person,indices,commands):
    truth=labels[indices]
    select=evaluation_mask(person)[indices]
    if not select.any():return None
    y=truth[select];pred=commands[select]
    active_mask=y>0;rest_mask=y==0
    f1_active=(f1_score(y[active_mask],pred[active_mask],labels=np.arange(1,53),
                        average='macro',zero_division=0) if active_mask.any() else 0.)
    f1_all=f1_score(y,pred,labels=np.arange(53),average='macro',zero_division=0)
    # Evénements actifs réellement annotés (même ceux n'ayant aucune fenêtre de prédiction).
    person_events=events[(events.subject==person)&(events.phase>=3)&(events.phase<=10)]
    active_events=person_events[person_events.gesture>0]
    rest_s=float(person_events.loc[person_events.gesture==0,'duration_s'].sum())
    # Un segment est détecté lorsque sa classe apparaît au moins une fois pendant ce segment.
    ids=event_ids[indices][select]
    since=since_s[indices][select]
    correct=(y>0)&(pred==y)
    hit_ids=ids[correct]
    first_latency={}
    for eid,lat in zip(hit_ids,since[correct]):
        eid=int(eid)
        if eid not in first_latency or lat<first_latency[eid]:first_latency[eid]=float(lat)
    detected=int(active_events.event_id.isin(first_latency.keys()).sum())
    event_recall=detected/len(active_events) if len(active_events) else np.nan
    times=[first_latency[int(e)] for e in active_events.event_id if int(e) in first_latency]
    # Compte les *entrées* dans une période de fausse commande pendant le repos.
    false_full=np.zeros(len(indices),dtype=bool)
    false_full[select]=rest_mask & (pred!=0)
    previous=np.r_[False,false_full[:-1]]
    session_start=np.r_[True,sessions[indices][1:]!=sessions[indices][:-1]]
    false_count=int(np.sum(false_full & (~previous | session_start)))
    false_hr=3600*false_count/rest_s if rest_s>0 else np.nan
    return dict(subject=int(person),n_eval_windows=int(select.sum()),n_active_events=len(active_events),
                n_detected_events=detected,event_recall=event_recall,
                active_macro_f1=float(f1_active),all_macro_f1=float(f1_all),
                window_accuracy=float(accuracy_score(y,pred)),
                rest_specificity=float(np.mean(pred[rest_mask]==0)) if rest_mask.any() else np.nan,
                false_command_periods=false_count,rest_hours=rest_s/3600,
                false_commands_per_rest_hour=false_hr,
                latency_median_s=float(np.median(times)) if times else np.nan,
                latency_p90_s=float(np.percentile(times,90)) if times else np.nan)

## 9 — Valider les réglages sur 19 personnes, **sans leurs labels pendant le replay**

Pour chaque fold, LDA apprend avec les autres personnes. Le kNN personnel ne reçoit au maximum que les répétitions 1–2 et quelques exemples par classe. Nous testons ensuite différents poids de calibration, longueurs de moyenne causale et seuils de confiance.

**Sélection multiobjectif explicite** : cible exploratoire ≤20 fausses commandes par heure de repos, avec rappel événementiel >10 %. Si aucune combinaison ne respecte cette cible, nous **affichons un avertissement** et choisissons seulement une stratégie *diagnostique* qui minimise les fausses commandes parmi celles détectant au moins 10 % des gestes. Ce seuil n'est pas une norme clinique et ne garantit aucune sécurité.

Une même personne pèse autant qu'une autre, quel que soit son nombre de fenêtres. Aucun ancien participant de test n'est utilisé.

In [ ]:
def all_candidates():
    result=[]
    for budget in CAL_BUDGETS:
        alphas=[0.0] if budget==0 else ALPHAS_PERSONAL
        for alpha in alphas:
            for k in SMOOTH_K:
                for threshold in CONF_THRESHOLDS:
                    result.append((budget,float(alpha),int(k),float(threshold)))
    return result

CANDIDATES=all_candidates()
print('Nombre de configurations:',len(CANDIDATES))
CV_ROWS=[]
start=time.time()
for fold,(ti,vi) in enumerate(GroupKFold(n_splits=RESEARCH_FOLDS).split(
        RESEARCH,groups=RESEARCH),1):
    train_people=RESEARCH[ti]; val_people=RESEARCH[vi]
    general=create_general_model(train_people)
    print('Fold',fold,'| recherche entraînement:',len(train_people),'| nouveaux utilisateurs:',val_people.tolist())
    for person in val_people:
        person=int(person)
        ix,pg,pp,info=get_scores(general,person)
        for budget,alpha,k,threshold in CANDIDATES:
            if budget and pp is None:continue
            p=(1-alpha)*pg+alpha*pp if budget else pg
            decisions=continuous_decisions(p,sessions[ix],k=k,threshold=threshold)
            scores=evaluate_commands(person,ix,decisions)
            if scores is not None:
                CV_ROWS.append(dict(fold=fold,budget=budget,alpha=alpha,k=k,threshold=threshold,
                                    n_cal=info['n_cal'] if budget else 0,**scores))
    del general
    print('Écoulé:',round(time.time()-start,1),'s')
cv_detail=pd.DataFrame(CV_ROWS)
assert cv_detail.subject.nunique()==19
cv_summary=(cv_detail.groupby(['budget','alpha','k','threshold'],as_index=False)
    .agg(mean_event_recall=('event_recall','mean'),sd_event_recall=('event_recall','std'),
         mean_active_f1=('active_macro_f1','mean'),mean_rest_specificity=('rest_specificity','mean'),
         mean_false_h=('false_commands_per_rest_hour','mean'),
         mean_latency_s=('latency_median_s','mean'),people=('subject','nunique')))
cv_summary['quality']=(0.55*cv_summary.mean_event_recall+0.45*cv_summary.mean_active_f1)
cv_summary=cv_summary.sort_values('quality',ascending=False).reset_index(drop=True)
assert cv_summary.people.min()==19
feasible=cv_summary[(cv_summary.mean_false_h<=20)&(cv_summary.mean_event_recall>=0.10)]
if len(feasible):
    selected=feasible.sort_values('quality',ascending=False).iloc[0].to_dict()
    selection_note='Cible exploratoire de 20 fausses commandes/h respectée en CV'
else:
    acceptable=cv_summary[cv_summary.mean_event_recall>=0.10]
    if acceptable.empty:acceptable=cv_summary
    lowest=acceptable.mean_false_h.min()
    options=acceptable[acceptable.mean_false_h<=lowest*1.10+0.01]
    selected=options.sort_values('quality',ascending=False).iloc[0].to_dict()
    selection_note='AVERTISSEMENT : aucune stratégie ne respecte le seuil de fausses commandes ; résultat DIAGNOSTIQUE, non utilisable pour une commande réelle'
print(selection_note)
print('Meilleure stratégie choisie par CV :',selected)
display(cv_summary.head(12).round(4))
cv_detail.to_csv(WORK_DIR/'mantis_v8_cv_by_subject.csv',index=False)
cv_summary.to_csv(WORK_DIR/'mantis_v8_cv_candidates.csv',index=False)

## 10 — Vérification exploratoire sur les 4 sujets de validation historiques

Leurs résultats ont déjà été consultés dans les versions précédentes : cette étape est une **réplication exploratoire**, pas un nouveau test indépendant. Les quatre anciens sujets de test (`12,13,16,25`) restent exclus. Nous comparons les mêmes réglages choisis en CV **avec** et **sans** calibration sur les mêmes endpoints éligibles.

**Limite du replay personnalisé :** le kNN personnel est ajusté sur toutes les répétitions 1–2 autorisées *avant* le replay. Si, dans le CSV, certaines de ces répétitions surviennent après une répétition 3–10 d'un autre geste, il s'agit d'un scénario *rétrospectif*, et non d'une acquisition chronologique prospective.

In [ ]:
general_final=create_general_model(RESEARCH)
CHECK_ROWS=[]
for person in HIST_VAL:
    person=int(person)
    ix,pg,pp,info=get_scores(general_final,person)
    for title,budget,alpha,k,thresh in [
        ('Sans calibration',0,0.0,selected['k'],selected['threshold']),
        ('Calibration personnalisée',int(selected['budget']),float(selected['alpha']),
         int(selected['k']),float(selected['threshold']))]:
        if budget and pp is None:continue
        p=(1-alpha)*pg+alpha*pp if budget else pg
        decision=continuous_decisions(p,sessions[ix],k=int(k),threshold=float(thresh))
        m=evaluate_commands(person,ix,decision)
        if m is not None:CHECK_ROWS.append(dict(method=title,**m))
check_detail=pd.DataFrame(CHECK_ROWS)
summary=(check_detail.groupby('method',as_index=False).agg(
    event_recall=('event_recall','mean'),active_macro_f1=('active_macro_f1','mean'),
    false_commands_per_rest_hour=('false_commands_per_rest_hour','mean'),
    rest_specificity=('rest_specificity','mean'),
    median_observation_s=('latency_median_s','median'),
    p90_observation_s=('latency_p90_s','median'),
    people=('subject','nunique')))
print('V8 — Vérification EXPLORATOIRE (anciens sujets de validation):')
display(summary.round(4))
display(check_detail[['method','subject','event_recall','false_commands_per_rest_hour',
                      'rest_specificity','latency_median_s','latency_p90_s']].round(3))
check_detail.to_csv(WORK_DIR/'mantis_v8_historical_validation_by_subject.csv',index=False)
summary.to_csv(WORK_DIR/'mantis_v8_historical_validation_summary.csv',index=False)

## 11 — Visualiser une prédiction continue, sans afficher les annotations au classifieur

Pour la figure uniquement, nous pouvons superposer **après prédiction** la vraie étiquette sur l'axe temporel. Cette annotation n'intervient ni dans `continuous_decisions` ni dans la moyenne causale.

Les identifiants de session permettent un reset seulement lors d'une discontinuité d'enregistrement ; un repos ou un changement de geste ne déclenche **aucun reset**.

In [ ]:
person=int(HIST_VAL[0]); ix,pg,pp,calinfo=get_scores(general_final,person)
alpha=float(selected['alpha']);budget=int(selected['budget'])
p=(1-alpha)*pg+alpha*pp if budget and pp is not None else pg
out=continuous_decisions(p,sessions[ix],k=int(selected['k']),threshold=float(selected['threshold']))
mask=evaluation_mask(person)[ix]
# Montrer une seule session, et uniquement les premières fenêtres éligibles.
idx=np.flatnonzero(mask)
if len(idx):
    sid=sessions[ix][idx[0]]
    sub_idx=idx[sessions[ix][idx]==sid][:200]
    t=(d['endpoint_row'][ix][sub_idx]-d['endpoint_row'][ix][sub_idx[0]])/SR
    fig,ax=plt.subplots(figsize=(13,4))
    ax.step(t,labels[ix][sub_idx],where='post',label='Annotation (après coup)',alpha=.6)
    ax.step(t,out[sub_idx],where='post',label='Commande prédite (en continu)',alpha=.85)
    ax.set(xlabel='Secondes dans cet extrait',ylabel='Geste (0 = repos)',
           title=f'V8 — Extrait continu, sujet {person} (sans réinitialisation par geste)')
    ax.legend();plt.tight_layout();plt.show()
else: print('Pas de fenêtres évaluables pour la figure.')

## 12 — Sauvegarde des résultats, paramètres et limites

Nous sauvegardons les métriques et le modèle général pour permettre de reproduire cette expérience. **Aucune application en temps réel n'est validée** par ce notebook : il faudra une acquisition physique, des données de repos bien annotées, une calibration chronologiquement séparée et une validation sur de nouvelles sessions avant de commander un actionneur.

In [ ]:
import joblib
settings={
    'version':'v8','dataset':DATASET_HANDLE,'raw_file_bytes':DATASET_PATH.stat().st_size,
    'source':'NinaPro DB1 - 10 EMG channels 100 Hz',
    'classes':53,'rest_label':0,'window_samples':WINDOW,'hop_samples':HOP,
    'features':FEATURE_NAMES,'research_subjects':RESEARCH.astype(int).tolist(),
    'historical_validation_subjects':HIST_VAL.astype(int).tolist(),
    'historic_test_subjects_excluded':HIST_TEST.astype(int).tolist(),
    'candidates_count':len(CANDIDATES),'selection_metric':'Priorité à moins de 20 fausses commandes/h si possible; sinon compromis diagnostique parmi stratégies avec rappel événementiel >=10 %',
    'selected_settings':{'budget':int(selected['budget']),'alpha':float(selected['alpha']),
                         'k':int(selected['k']),'threshold':float(selected['threshold'])},
    'active_stability':ACTIVE_STABILITY,'rest_stability':REST_STABILITY,
    'limitations':['offline replay, not hardware latency','personal calibration retrospective by repetitions',
                   'legacy validation previously inspected','labels used for metrics only, not inference',
                   'exercise/session boundaries can reset model, not gesture boundaries'],
}
with open(WORK_DIR/'mantis_v8_metadata.json','w',encoding='utf-8') as fp:
    json.dump(settings,fp,ensure_ascii=False,indent=2)
joblib.dump(general_final,WORK_DIR/'mantis_v8_general_lda.joblib')
print('V8: résultats et modèle général sauvegardés dans',WORK_DIR)
print('Fichiers: CV, validation, événements, cache, modèle et métadonnées')

## 13 — Interpréter la v8 avant d'envisager un prototype

- Une **accuracy de fenêtre** élevée peut masquer de nombreuses fausses commandes au repos.
- Un modèle silencieux peut avoir **0 fausse commande**, mais **0 geste détecté** : toujours lire *les deux* métriques.
- Le délai affiché est un **temps d'observation annoté**, non une latence de calcul ni une latence de commande d'un exosquelette.
- Les réinitialisations se font aux **sessions enregistrées** (changement sujet/exercice ou discontinuité), pas aux gestes. NinaPro peut cependant contenir des pauses artificielles entre fichiers ; un test sur un vrai flux capteur reste indispensable.
- La calibration par répétitions est **rétrospective**, car les sessions originales n'étaient pas nécessairement organisées pour une calibration réalisée intégralement avant le test. Le prochain protocole réellement prospectif exigera des acquisitions spécifiques.
- Les données de repos ne sont pas forcément uniformément échantillonnées dans les exercices, et `restimulus=0` n'est pas une annotation clinique d'immobilité parfaitement précise.
- La classe repos est incluse dans le classifieur : **53 classes au total**.

**Succès d'ingénierie pour une v9 sur matériel :** acquérir une vraie séquence `repos → geste → repos`, calibrer sur une portion chronologiquement antérieure, vérifier les déclenchements non voulus, le temps de réponse, les réorientations d'électrodes et la reproductibilité entre sessions.